
Part A: Classical Machine Learning - The Baseline
Objective: Use traditional machine learning to perform sentiment analysis on a text dataset (IMDb Reviews,
Twitter Sentiment, Amazon Reviews, or any suitable dataset), establishing the classical baselines every later
experiment is measured against.



In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [2]:
path='/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv'
IMDB=pd.DataFrame(pd.read_csv(path))
IMDB.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
#inspect columns
print(IMDB.shape)
print(IMDB.columns)
print(IMDB["sentiment"].value_counts())
print(IMDB["sentiment"])


(50000, 2)
Index(['review', 'sentiment'], dtype='object')
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
0        positive
1        positive
2        positive
3        negative
4        positive
           ...   
49995    positive
49996    negative
49997    negative
49998    negative
49999    negative
Name: sentiment, Length: 50000, dtype: object


In [4]:
IMDB["sentiment"] = IMDB["sentiment"].map({"positive": 1, "negative": 0})
print(IMDB["sentiment"])

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 50000, dtype: int64


In [5]:
import re
def clean(text):
  text=text.lower()
  text=re.sub('<.*?>',' ',text)
  text=re.sub('[^a-z]+',' ',text)
  text=re.sub(r'\s+',' ',text).strip()
  return text

In [6]:
IMDB["clean_review"]=IMDB["review"].apply(clean)
print(IMDB[["review","clean_review"]])

                                                  review  \
0      One of the other reviewers has mentioned that ...   
1      A wonderful little production. <br /><br />The...   
2      I thought this was a wonderful way to spend ti...   
3      Basically there's a family where a little boy ...   
4      Petter Mattei's "Love in the Time of Money" is...   
...                                                  ...   
49995  I thought this movie did a down right good job...   
49996  Bad plot, bad dialogue, bad acting, idiotic di...   
49997  I am a Catholic taught in parochial elementary...   
49998  I'm going to have to disagree with the previou...   
49999  No one expects the Star Trek movies to be high...   

                                            clean_review  
0      one of the other reviewers has mentioned that ...  
1      a wonderful little production the filming tech...  
2      i thought this was a wonderful way to spend ti...  
3      basically there s a family where a l

In [7]:
x=IMDB["clean_review"]
y=IMDB["sentiment"]
#Train and test
from sklearn.model_selection import train_test_split
x_train,x_temp,y_train,y_temp=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

x_val,x_test,y_val,y_test=train_test_split(x_temp,y_temp,test_size=0.5,random_state=42,stratify=y_temp)

print("X Train shape: ",x_train.shape)
print("X val: ",x_val.shape)
print("x test: ",x_test.shape)

X Train shape:  (40000,)
X val:  (5000,)
x test:  (5000,)


In [8]:
print("Train class counts:\n", y_train.value_counts())
print("Val class counts:\n", y_val.value_counts())
print("Test class counts:\n", y_test.value_counts())

Train class counts:
 sentiment
1    20000
0    20000
Name: count, dtype: int64
Val class counts:
 sentiment
0    2500
1    2500
Name: count, dtype: int64
Test class counts:
 sentiment
0    2500
1    2500
Name: count, dtype: int64


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.95
)

In [10]:
x_train_tfidf=tfidf.fit_transform(x_train)
x_val_tfidf=tfidf.transform(x_val)
x_test_tfidf=tfidf.transform(x_test)
#printing statements
print("Train TF-IDF shape: ",x_train_tfidf.shape)
print("val TF-IDF shape: ",x_val_tfidf.shape)
print("test TF-IDF shape: ",x_test_tfidf.shape)
print("Vocab size:", len(tfidf.vocabulary_))

Train TF-IDF shape:  (40000, 20000)
val TF-IDF shape:  (5000, 20000)
test TF-IDF shape:  (5000, 20000)
Vocab size: 20000


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score
logistic=LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)


In [12]:
logistic.fit(x_train_tfidf, y_train)

y_val_pred_log = logistic.predict(x_val_tfidf)
print("Logistic Regression – Validation accuracy:",
      accuracy_score(y_val, y_val_pred_log))

print("\nLogistic Regression – Validation classification report:")
print(classification_report(y_val, y_val_pred_log))

Logistic Regression – Validation accuracy: 0.9032

Logistic Regression – Validation classification report:
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      2500
           1       0.89      0.92      0.90      2500

    accuracy                           0.90      5000
   macro avg       0.90      0.90      0.90      5000
weighted avg       0.90      0.90      0.90      5000



In [13]:
from sklearn.svm import LinearSVC

svm_clf = LinearSVC(
    C=1.0,
    random_state=42,
)

svm_clf.fit(x_train_tfidf, y_train)

y_val_pred_svm = svm_clf.predict(x_val_tfidf)
print("Linear SVM – Validation accuracy:",
      accuracy_score(y_val, y_val_pred_svm))

print("\nLinear SVM – Validation classification report:")
print(classification_report(y_val, y_val_pred_svm))

Linear SVM – Validation accuracy: 0.9004

Linear SVM – Validation classification report:
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      2500
           1       0.89      0.91      0.90      2500

    accuracy                           0.90      5000
   macro avg       0.90      0.90      0.90      5000
weighted avg       0.90      0.90      0.90      5000



In [14]:
# Logistic Regression on test
y_test_pred_log = logistic.predict(x_test_tfidf)
print("Logistic Regression – Test accuracy:",
      accuracy_score(y_test, y_test_pred_log))
print("\nLogistic Regression – Test classification report:")
print(classification_report(y_test, y_test_pred_log))

# SVM on test
y_test_pred_svm = svm_clf.predict(x_test_tfidf)
print("Linear SVM – Test accuracy:",
      accuracy_score(y_test, y_test_pred_svm))
print("\nLinear SVM – Test classification report:")
print(classification_report(y_test, y_test_pred_svm))

Logistic Regression – Test accuracy: 0.9082

Logistic Regression – Test classification report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91      2500
           1       0.91      0.91      0.91      2500

    accuracy                           0.91      5000
   macro avg       0.91      0.91      0.91      5000
weighted avg       0.91      0.91      0.91      5000

Linear SVM – Test accuracy: 0.9106

Linear SVM – Test classification report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91      2500
           1       0.91      0.91      0.91      2500

    accuracy                           0.91      5000
   macro avg       0.91      0.91      0.91      5000
weighted avg       0.91      0.91      0.91      5000



In [15]:
# choose one model, e.g. SVM
import numpy as np

mis_idx = np.where(y_test != y_test_pred_svm)[0]
print("Number of misclassified test examples (SVM):", len(mis_idx))

# show first 5
for i in mis_idx[:5]:
    print("----")
    print("True label:", y_test.iloc[i])
    print("Predicted:", y_test_pred_svm[i])
    print("Review:", x_test.iloc[i][:500])  # first 500 chars


Number of misclassified test examples (SVM): 447
----
True label: 1
Predicted: 0
Review: this movie is pretty cheesy but i do give it credit for at least trying to provide some characterization for it s principles there are some great moments in the film and the dialogue has some great moments as well the aerial assault sequence is perhaps the best part of the movie i guess i really like the idea of what lengths a veteran will go for a fellow veteran sure it s not all that well done but the premise is not at all bad tom
----
True label: 0
Predicted: 1
Review: no surprise except in how quickly abc reacted to the dismal ratings according to published reports variety the show garnered the worst ratings in the history of the abc television network and i quote abc s music talent competition the one opened tuesday night to cancel me now ratings the article went on to say that the show received a shockingly low rating share in adult and million viewers overall that makes it the weakest premie